# LoRA Hyperparameter Sweep — BioCLIP 2 on DiopSIS

This notebook runs a hyperparameter sweep for LoRA fine-tuning of BioCLIP 2's image encoder on the DiopSIS alpine insect dataset (14 orders, ~40k crops).

**Setup:** single stratified 80/20 train/val split. Linear probe (logistic regression) on top of LoRA-modified image features.

**Outputs:**
- `output_sweep/sweep_summary.csv` — one row per config, aggregate metrics.
- `output_sweep/sweep_per_class.csv` — per-class precision/recall/F1 for every config.

**Hardware:** A100 80GB SXM on RunPod (CUDA).

---

## What we're comparing

The sweep tests how LoRA hyperparameters (learning rate, rank, epochs) affect downstream classification. The frozen-features baseline (no LoRA) is included as a reference.

For each config, image features are extracted and a multinomial logistic regression classifier is fit on the training set, then evaluated on the held-out validation set.

## Imports and configuration

In [1]:
import re
import time
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import normalize as sk_normalize
from torch.utils.data import Dataset, DataLoader
import open_clip
from peft import LoraConfig, get_peft_model

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch: 2.8.0+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB


In [2]:
# Configuration — paths and hyperparameters
CONFIG = {
    "model_name": "hf-hub:imageomics/bioclip-2",
    "embedding_dim": 768,
    "device": "cuda" if torch.cuda.is_available() else "cpu",

    "meta_path": "/workspace/thesis/ultimate_bioclip_top1_rank_order_meta_12062025.csv",
    "image_root": "/workspace/thesis/ALL_copy",
    "output_dir": "/workspace/thesis/output_sweep",

    "image_id_col": "image_id",
    "label_col": "label",

    # Train/val split
    "val_size": 0.2,
    "random_state": 0,
    "min_class_size": 50,  # classes below this excluded from LoRA training

    # LoRA structural parameters
    "lora_target_modules": ["out_proj", "c_fc", "c_proj"],
    "lora_alpha_factor": 2,  # alpha = r * factor
    "lora_dropout": 0.05,

    # Training
    "batch_size": 64,
    "weight_decay": 1e-4,
    "num_workers": 4,

    # Linear probe
    "linear_probe_C": 10.0,
    "linear_probe_max_iter": 5000,

    # Hyperparameter grid: (name, lr, rank, epochs)
    "configs_to_test": [
        ("baseline_no_lora", None, 0,  0),
        ("lr1e4_r8_e2",   1e-4,  8,  2),
        ("lr1e4_r8_e3",   1e-4,  8,  3),
        ("lr5e4_r8_e2",   5e-4,  8,  2),
        ("lr5e4_r8_e3",   5e-4,  8,  3),
        ("lr1e3_r8_e2",   1e-3,  8,  2),
        ("lr5e4_r4_e2",   5e-4,  4,  2),
        ("lr5e4_r16_e2",  5e-4, 16,  2),
        ("lr1e4_r16_e3",  1e-4, 16,  3),
    ],
}

# Regex for extracting image_id from filenames like "Diptera_20240820235954_crop_2.jpg"
CROP_ID_PATTERN = re.compile(r"(\d{14}_crop_[a-zA-Z0-9]+\.jpg)$")

Path(CONFIG["output_dir"]).mkdir(parents=True, exist_ok=True)
print(f"Output directory: {CONFIG['output_dir']}")

Output directory: /workspace/thesis/output_sweep


## Helper functions

- `build_path_index`: scans the local image directory, maps each `image_id` to its full path.
- `ImageOrderDataset`: PyTorch Dataset that loads and preprocesses images on demand.
- `train_lora`: applies LoRA to the visual encoder and trains it with a classification head.
- `extract_features`: extracts L2-normalized image embeddings from a (possibly LoRA-modified) encoder.
- `evaluate_linear_probe`: fits a multinomial logistic regression and returns the classification report.

In [3]:
def build_path_index(image_root):
    """Scan local image directory, build {image_id: full_path} dictionary."""
    image_root = Path(image_root)
    print(f"Scanning {image_root}...")
    files = list(image_root.glob("*.jpg"))
    print(f"  Found {len(files)} JPGs")
    path_index = {}
    for f in files:
        m = CROP_ID_PATTERN.search(f.name)
        if m:
            path_index[m.group(1)] = str(f)
    print(f"  Mapped {len(path_index)} image_ids to paths")
    return path_index


class ImageOrderDataset(Dataset):
    """Dataset of (image, label) pairs loaded from disk."""
    def __init__(self, image_paths, labels, preprocess):
        self.image_paths = image_paths
        self.labels = torch.tensor(labels, dtype=torch.long)
        self.preprocess = preprocess

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("RGB")
        img = self.preprocess(img)
        return img, self.labels[idx]

In [4]:
def train_lora(model, train_loader, classnames, lr, r, epochs, config):
    """Apply LoRA to the visual encoder and train it with a classification head.

    Returns the LoRA-wrapped visual encoder (the head is discarded — only used
    to provide a training signal).
    """
    device = config["device"]
    n_classes = len(classnames)

    lora_config = LoraConfig(
        r=r,
        lora_alpha=r * config["lora_alpha_factor"],
        lora_dropout=config["lora_dropout"],
        target_modules=config["lora_target_modules"],
        bias="none",
    )
    visual = get_peft_model(model.visual, lora_config).to(device)
    visual.print_trainable_parameters()

    head = nn.Linear(config["embedding_dim"], n_classes).to(device)
    trainable = [p for p in visual.parameters() if p.requires_grad] + list(head.parameters())

    optimizer = torch.optim.AdamW(trainable, lr=lr, weight_decay=config["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    for epoch in range(epochs):
        visual.train()
        head.train()
        total_loss, n_batches = 0.0, 0
        ep_start = time.time()
        for batch_idx, (imgs, labels) in enumerate(train_loader):
            imgs = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            features = F.normalize(visual(imgs), dim=-1)
            logits = head(features)
            loss = F.cross_entropy(logits, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            n_batches += 1
            if batch_idx % 100 == 0:
                print(f"      Ep {epoch+1} batch {batch_idx}/{len(train_loader)}: loss={loss.item():.4f}")
        scheduler.step()
        print(f"      Ep {epoch+1}: avg loss={total_loss/n_batches:.4f} "
              f"({(time.time()-ep_start)/60:.1f} min)")

    return visual


def extract_features(encoder_or_model, dataset, config, is_lora=True):
    """Extract L2-normalized image features. If is_lora=False, uses model.visual directly."""
    device = config["device"]
    encoder = encoder_or_model if is_lora else encoder_or_model.visual
    encoder.eval()
    loader = DataLoader(
        dataset, batch_size=config["batch_size"], shuffle=False,
        num_workers=config["num_workers"], pin_memory=True,
    )
    all_features = []
    with torch.no_grad():
        for imgs, _ in loader:
            features = F.normalize(encoder(imgs.to(device, non_blocking=True)), dim=-1)
            all_features.append(features.cpu().numpy())
    return np.concatenate(all_features, axis=0)


def evaluate_linear_probe(train_features, train_labels, val_features, val_labels, classnames, config):
    """Fit logistic regression on training features, evaluate on validation set."""
    clf = LogisticRegression(
        C=config["linear_probe_C"],
        max_iter=config["linear_probe_max_iter"],
        solver="lbfgs",
        random_state=config["random_state"],
    )
    clf.fit(train_features, train_labels)
    preds = clf.predict(val_features)
    report = classification_report(
        val_labels, preds, labels=list(range(len(classnames))),
        target_names=classnames, output_dict=True, zero_division=0,
    )
    return report, preds

## Loading model and data

In [5]:
# Loading BioCLIP 2
print("Loading BioCLIP 2...")
model, _, preprocess = open_clip.create_model_and_transforms(CONFIG["model_name"])
model = model.to(CONFIG["device"])
print("Done.")

Loading BioCLIP 2...


open_clip_config.json:   0%|          | 0.00/534 [00:00<?, ?B/s]

open_clip_model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

Done.


In [6]:
# Building path index from local image directory and merging with metadata
path_index = build_path_index(CONFIG["image_root"])

meta = pd.read_csv(CONFIG["meta_path"])
meta = meta.dropna(subset=[CONFIG["label_col"], CONFIG["image_id_col"]]).reset_index(drop=True)
meta["local_path"] = meta[CONFIG["image_id_col"]].map(path_index)
matched = meta.dropna(subset=["local_path"]).reset_index(drop=True)
print(f"Matched: {len(matched)} crops with valid labels and local paths")

# Setting up label indexing
y_str = matched[CONFIG["label_col"]].to_numpy()
image_paths = matched["local_path"].to_numpy()
classnames = sorted(np.unique(y_str).tolist())
class_to_idx = {c: i for i, c in enumerate(classnames)}
y = np.array([class_to_idx[s] for s in y_str])

class_counts = pd.Series(y_str).value_counts()
eligible_classes = class_counts[class_counts >= CONFIG["min_class_size"]].index.tolist()
eligible_class_indices = [class_to_idx[c] for c in eligible_classes]

print(f"\nClasses ({len(classnames)}): {classnames}")
print(f"Eligible for LoRA training (n>=50): {eligible_classes}")
print(f"\nClass counts:")
for c in classnames:
    print(f"  {c}: {int((y_str == c).sum())}")

Scanning /workspace/thesis/ALL_copy...
  Found 48096 JPGs
  Mapped 48095 image_ids to paths
Matched: 39787 crops with valid labels and local paths

Classes (14): ['Araneae', 'Blattodea', 'Coleoptera', 'Diptera', 'Hemiptera', 'Hymenoptera', 'Ixodida', 'Lepidoptera', 'Mecoptera', 'Neuroptera', 'Orthoptera', 'Plecoptera', 'Raphidioptera', 'Trichoptera']
Eligible for LoRA training (n>=50): ['Diptera', 'Hymenoptera', 'Lepidoptera', 'Coleoptera', 'Hemiptera', 'Araneae', 'Neuroptera', 'Trichoptera', 'Plecoptera']

Class counts:
  Araneae: 564
  Blattodea: 23
  Coleoptera: 1143
  Diptera: 31913
  Hemiptera: 775
  Hymenoptera: 3326
  Ixodida: 1
  Lepidoptera: 1744
  Mecoptera: 34
  Neuroptera: 115
  Orthoptera: 8
  Plecoptera: 54
  Raphidioptera: 2
  Trichoptera: 85


In [7]:
# Stratified train/val split, handling singleton classes
class_counts_arr = np.bincount(y, minlength=len(classnames))
splittable_mask = class_counts_arr[y] >= 2
splittable_indices = np.where(splittable_mask)[0]
unsplittable_indices = np.where(~splittable_mask)[0]

train_idx_split, val_idx = train_test_split(
    splittable_indices,
    test_size=CONFIG["val_size"],
    stratify=y[splittable_indices],
    random_state=CONFIG["random_state"],
)
# Singletons go entirely to training
train_idx = np.concatenate([train_idx_split, unsplittable_indices])

# Filtering training set to eligible classes for LoRA training
# (linear probe still uses full training set across all 14 classes)
train_mask = np.isin(y[train_idx], eligible_class_indices)
train_idx_eligible = train_idx[train_mask]

train_paths_eligible = image_paths[train_idx_eligible]
train_labels_eligible = y[train_idx_eligible]
train_paths_full = image_paths[train_idx]
train_labels_full = y[train_idx]
val_paths = image_paths[val_idx]
val_labels = y[val_idx]

train_dataset_eligible = ImageOrderDataset(train_paths_eligible, train_labels_eligible, preprocess)
train_dataset_full = ImageOrderDataset(train_paths_full, train_labels_full, preprocess)
val_dataset = ImageOrderDataset(val_paths, val_labels, preprocess)

train_loader = DataLoader(
    train_dataset_eligible, batch_size=CONFIG["batch_size"],
    shuffle=True, num_workers=CONFIG["num_workers"], pin_memory=True,
)

print(f"Train (LoRA, eligible only): {len(train_idx_eligible)}")
print(f"Train (full, for probe):     {len(train_idx)}")
print(f"Val (all 14 classes):        {len(val_idx)}")

Train (LoRA, eligible only): 31774
Train (full, for probe):     31829
Val (all 14 classes):        7958


## Running the sweep

For each configuration:
1. (If `r=0`) Use frozen BioCLIP 2 image features as baseline.
2. (Otherwise) Reload a fresh BioCLIP 2 model, apply LoRA, train for the specified epochs.
3. Extract L2-normalized image features from the (modified) encoder.
4. Fit a logistic regression linear probe on training features, evaluate on validation.

Results are saved incrementally to disk after each config completes (this approach survives interruptions).

In [8]:
summary_rows = []
per_class_rows = []
overall_start = time.time()

for config_name, lr, r, epochs in CONFIG["configs_to_test"]:
    print(f"\n{'='*70}")
    print(f"Config: {config_name} (lr={lr}, r={r}, epochs={epochs})")
    print(f"{'='*70}")
    config_start = time.time()

    if r == 0:
        # Baseline: no LoRA, just frozen features
        print("  Frozen baseline — extracting features...")
        train_features = extract_features(model, train_dataset_full, CONFIG, is_lora=False)
        val_features = extract_features(model, val_dataset, CONFIG, is_lora=False)
        train_duration = 0.0
    else:
        # Reloading fresh model for clean LoRA application (avoid adapter stacking)
        print("  Reloading fresh BioCLIP 2...")
        model_fresh, _, _ = open_clip.create_model_and_transforms(CONFIG["model_name"])
        model_fresh = model_fresh.to(CONFIG["device"])

        train_start = time.time()
        visual = train_lora(model_fresh, train_loader, classnames, lr, r, epochs, CONFIG)
        train_duration = time.time() - train_start

        print("  Extracting LoRA features...")
        train_features = extract_features(visual, train_dataset_full, CONFIG, is_lora=True)
        val_features = extract_features(visual, val_dataset, CONFIG, is_lora=True)

        del visual, model_fresh
        torch.cuda.empty_cache()

    # Normalizing features (matches CLIP convention)
    train_features = sk_normalize(train_features, norm="l2", axis=1)
    val_features = sk_normalize(val_features, norm="l2", axis=1)

    # Linear probe evaluation
    report, _ = evaluate_linear_probe(
        train_features, train_labels_full, val_features, val_labels, classnames, CONFIG,
    )

    # Extracting aggregate metrics
    weighted_f1 = report["weighted avg"]["f1-score"]
    macro_f1_all = report["macro avg"]["f1-score"]
    eligible_f1s = [report[c]["f1-score"] for c in eligible_classes if c in report]
    macro_f1_restricted = float(np.mean(eligible_f1s)) if eligible_f1s else 0.0
    hemiptera_f1 = report.get("Hemiptera", {}).get("f1-score", 0.0)
    coleoptera_f1 = report.get("Coleoptera", {}).get("f1-score", 0.0)

    config_duration = time.time() - config_start

    summary_rows.append({
        "config": config_name, "lr": lr, "r": r, "epochs": epochs,
        "train_time_sec": int(train_duration),
        "total_time_sec": int(config_duration),
        "weighted_f1": round(weighted_f1, 4),
        "macro_f1_all": round(macro_f1_all, 4),
        "macro_f1_restricted_n50": round(macro_f1_restricted, 4),
        "Hemiptera_f1": round(hemiptera_f1, 4),
        "Coleoptera_f1": round(coleoptera_f1, 4),
    })

    for cls_name in classnames:
        stats = report.get(cls_name, {})
        per_class_rows.append({
            "config": config_name, "class": cls_name,
            "precision": round(stats.get("precision", 0.0), 4),
            "recall": round(stats.get("recall", 0.0), 4),
            "f1": round(stats.get("f1-score", 0.0), 4),
            "support": int(stats.get("support", 0)),
            "trained_in_lora": cls_name in eligible_classes,
        })

    # Saving intermediate results (survives crashes)
    pd.DataFrame(summary_rows).to_csv(
        Path(CONFIG["output_dir"]) / "sweep_summary.csv", index=False
    )
    pd.DataFrame(per_class_rows).to_csv(
        Path(CONFIG["output_dir"]) / "sweep_per_class.csv", index=False
    )

    print(f"\n  weighted={weighted_f1:.4f}, macro(n>=50)={macro_f1_restricted:.4f}, "
          f"Hemiptera={hemiptera_f1:.4f}, Coleoptera={coleoptera_f1:.4f}")
    print(f"  Time: {timedelta(seconds=int(config_duration))}")

print(f"\n{'='*70}")
print(f"SWEEP COMPLETE — Total time: {timedelta(seconds=int(time.time() - overall_start))}")
print(f"{'='*70}")


Config: baseline_no_lora (lr=None, r=0, epochs=0)
  Frozen baseline — extracting features...

  weighted=0.9694, macro(n>=50)=0.8830, Hemiptera=0.7380, Coleoptera=0.8205
  Time: 0:06:37

Config: lr1e4_r8_e2 (lr=0.0001, r=8, epochs=2)
  Reloading fresh BioCLIP 2...
trainable params: 2,359,296 || all params: 306,325,504 || trainable%: 0.7702
      Ep 1 batch 0/497: loss=2.6328
      Ep 1 batch 100/497: loss=1.9172
      Ep 1 batch 200/497: loss=1.6249
      Ep 1 batch 300/497: loss=1.2561
      Ep 1 batch 400/497: loss=1.0244
      Ep 1: avg loss=1.5300 (13.8 min)
      Ep 2 batch 0/497: loss=0.9582
      Ep 2 batch 100/497: loss=0.8284
      Ep 2 batch 200/497: loss=0.8261
      Ep 2 batch 300/497: loss=0.7294
      Ep 2 batch 400/497: loss=0.6725
      Ep 2: avg loss=0.7217 (14.0 min)
  Extracting LoRA features...

  weighted=0.9786, macro(n>=50)=0.9068, Hemiptera=0.8580, Coleoptera=0.8864
  Time: 0:35:24

Config: lr1e4_r8_e3 (lr=0.0001, r=8, epochs=3)
  Reloading fresh BioCLIP 2...
t

## Results summary

In [9]:
summary_df = pd.DataFrame(summary_rows)
summary_df = summary_df.sort_values("macro_f1_restricted_n50", ascending=False)
print("Summary (sorted by macro F1 restricted to n>=50):\n")
summary_df

Summary (sorted by macro F1 restricted to n>=50):



,config,lr,r,epochs,train_time_sec,total_time_sec,weighted_f1,macro_f1_all,macro_f1_restricted_n50,Hemiptera_f1,Coleoptera_f1
7,lr5e4_r16_e2,0.0005,16,2,1651,2126,0.9811,0.6454,0.9299,0.9020,0.8967
4,lr5e4_r8_e3,0.0005,8,3,2481,2935,0.9820,0.7146,0.9172,0.9148,0.8938
5,lr1e3_r8_e2,0.0010,8,2,1661,2117,0.9804,0.7121,0.9133,0.9097,0.8933
3,lr5e4_r8_e2,0.0005,8,2,1653,2149,0.9806,0.6488,0.9105,0.9150,0.8865
8,lr1e4_r16_e3,0.0001,16,3,2489,2964,0.9808,0.7182,0.9074,0.8932,0.8933
1,lr1e4_r8_e2,0.0001,8,2,1668,2124,0.9786,0.6401,0.9068,0.8580,0.8864
6,lr5e4_r4_e2,0.0005,4,2,1658,2125,0.9816,0.7067,0.9049,0.9137,0.8962
2,lr1e4_r8_e3,0.0001,8,3,2503,2982,0.9787,0.6290,0.8951,0.8629,0.8869
0,baseline_no_lora,NaN,0,0,0,397,0.9694,0.7343,0.8830,0.7380,0.8205


In [10]:
# Identifying winning config
winner = summary_df.iloc[0]
print(f"Winning config: {winner['config']}")
print(f"  lr = {winner['lr']}")
print(f"  r = {winner['r']}")
print(f"  epochs = {winner['epochs']}")
print(f"  weighted F1 = {winner['weighted_f1']}")
print(f"  macro F1 (n>=50) = {winner['macro_f1_restricted_n50']}")
print(f"  Hemiptera F1 = {winner['Hemiptera_f1']}")
print(f"  Coleoptera F1 = {winner['Coleoptera_f1']}")
print(f"\nUsing these hyperparameters in the following CV notebooks (02_cv_lora.ipynb and 03_cv_joint.ipynb).")


Winning config: lr5e4_r16_e2
  lr = 0.0005
  r = 16
  epochs = 2
  weighted F1 = 0.9811
  macro F1 (n>=50) = 0.9299
  Hemiptera F1 = 0.902
  Coleoptera F1 = 0.8967

Using these hyperparameters in the following CV notebooks (02_cv_lora.ipynb and 03_cv_joint.ipynb).


## Output files summary

The notebook generates:

| File | Purpose |
|---|---|
| `sweep_summary.csv` | One row per config: aggregate metrics (weighted F1, macro F1, Hemiptera F1, Coleoptera F1) and timing |
| `sweep_per_class.csv` | Per-class precision/recall/F1 for every config (14 classes × N configs rows) |

The `sweep_summary.csv` is used to identify the winning hyperparameter configuration. Sorted by `macro_f1_restricted_n50` (descending) and the top row gives the values to plug into notebooks `02_cv_lora.ipynb` and `03_cv_joint.ipynb`.